# DSCM Handler

In [ ]:
import subprocess
# LEAVE IN PLACEHOLDER subprocess.run("aws codeartifact login #########", shell=True, check=True)

packages = ['pandas', 'geopandas', 'numpy<=2.3', 'lipbysal', 'boto3', 'scikit-learn', 'shapely', 'xgboost', 'matplotlib', 'seaborn']
for package in packages:
    subprocess.run(["pip", "install", package], check=True)

from functions import training_functions as tf
import pandas as pd
import time

In [ ]:
# Variables
period = "a25"
state = "SA"
pool = "all"
create_training_data = "Y"
train_model = "Y"
class_names = ['OTHER', 'x11', 'x21', 'x22', 'x31', 'x32', 'x33', 'x34', 'x35', 'x91', 'x92', 'x93', 'x94']
method = 'building_target'
extended_features = True
collapse_other_classes = True  # False keeps x35/x91-x94 as separate classes
adhoc_output = None  # e.g. "adhoc/2026-10-23_test" - writes all outputs there instead of the quarter folders

In [ ]:
from typing import Any


tf.set_output_location(adhoc_output)
tf.start_log(state)
if create_training_data == "Y":
    start_time = time.perf_counter()
    tf.generate_training_data_workflow(period, state, pool, method, extended_features)
    create_training_data = "N"
    print(f"Full time for training data generation {(time.perf_counter() - start_time):.4f} seconds.")
if train_model == "Y":
    start_time = time.perf_counter()
    model, X_val, y_val, le, class_set = tf.train_test_model(period, state, method, collapse_other_classes)
    print(f"Full time for training model {(time.perf_counter() - start_time):.4f} seconds.")
    start_time = time.perf_counter()
    model_eval_output = tf.complete_model_evaluation(
        period, 
        state,
        method,
        model,
        X_val.drop(columns='ARID'),
        y_val[['DWELLING_STRUCT']],
        class_names=list(filter(lambda item: item in class_set, class_names))
    )
    print(f"Full time for obtaining model evaluation {(time.perf_counter() - start_time):.4f} seconds.")
    start_time = time.perf_counter()
    tf.model_test_predictions(period, state, method, model_eval_output, X_val, y_val, class_names = list(filter(lambda item: item in class_set, class_names)), le=le)
    print(f"Full time for obtaining model test _predictions {(time.perf_counter() - start_time):.4f} seconds.")
    train_model == "N"
start_time = time.perf_counter()
tf.predict_candidates(period, state, method, model, class_set=list(filter(lambda item: item in class_set, class_names)), pool=pool)
print(f"Full time for obtaining model candidate predictions {(time.perf_counter() - start_time):.4f} seconds.")